# IterGen on G4: the 13 lost tasks after the fixes, and the helper repair

**Runtime: G4** (the GPU of the report's runs) runs both parts. **If G4 does not connect, pick L4**:
then only Part 2 runs, since Part 1 has to be on the report's GPU. Then **Runtime > Run all**.
Nothing to edit. The runtime disconnects itself at the end.

Code commit 3836cb8. Three IterGen versions:
- **v1** the report (commit 2c14eef, G4): read from Drive, not rerun
- **v2** the three fixes (ace68e3 strings, fa1fa66 unfinished regex, 1c9a4c5 braces in strings): modes `itergen_off`, `itergen_on`
- **v3** v2 + the helper repair: mode `itergen_on_repair`. A function the code calls but never defines is no
  longer backtracked; once the function is finished, the model writes the helper inside it.

Part 1 runs v2 (checks off) on the 13 tasks IterGen lost in the report, on the same GPU as the report.
The earlier L4 rerun differed from G4 on mbpp_604 and Qwen3.5 mbpp_230 for GPU reasons.
Part 2 runs v2 and v3 (checks on), on the same GPU, on the tasks where the replay shows the repair
changes something. Part 2 compares v2 with v3, so any GPU will do. On a GPU other than G4, the
"unconstr." and "v1 report" columns come from G4 and are shown only for reference.
Output on Drive: `final/<model>/<mode>_<GPU>_3836cb8/`.


In [ ]:
REPO_URL = 'https://github.com/BowenX307/constrained-llm-js-codegen-final.git'
COMMIT   = '3836cb88dd5dfbc9d48ee3dace44f44222093fe6'
REPORT_TAG = '2c14eef'          # the report's runs (G4), read from Drive
# Part 1: the tasks IterGen lost in the report (class: A string, B1 unfinished regex, B2 regex the grammar rejects)
LOST = {
    'Qwen/Qwen2.5-3B-Instruct': {'mbpp_230_replace_blank': 'A', 'mbpp_604_reverse_words': 'A',
                                 'mbpp_628_replace_spaces': 'A', 'mbpp_757_count_reverse_pairs': 'A',
                                 'mbpp_771_check_expression': 'A', 'mbpp_772_remove_length': 'A',
                                 'mbpp_285_text_match_two_three': 'B1', 'mbpp_787_text_match_three': 'B1'},
    'Qwen/Qwen3.5-4B':          {'mbpp_565_split': 'A', 'mbpp_619_move_num': 'A',
                                 'mbpp_757_count_reverse_pairs': 'A', 'mbpp_285_text_match_two_three': 'B1',
                                 'mbpp_230_replace_blank': 'B2'},
}
# Part 2: the tasks where the helper repair changes what IterGen does (from the offline replay)
HELPER = {
    "Qwen/Qwen2.5-3B-Instruct": [
        "mbpp_141_pancake_sort",
        "mbpp_160_find_solution",
        "mbpp_255_combinations_colors",
        "mbpp_256_count_Primes_nums",
        "mbpp_274_even_binomial_Coeff_Sum",
        "mbpp_462_combinations_list",
        "mbpp_592_sum_Of_product",
        "mbpp_608_bell_Number",
        "mbpp_635_heap_sort"
    ],
    "Qwen/Qwen3.5-4B": [
        "mbpp_123_amicable_numbers_sum",
        "mbpp_274_even_binomial_Coeff_Sum",
        "mbpp_592_sum_Of_product",
        "mbpp_635_heap_sort"
    ]
}
AUTO_DISCONNECT = True


In [ ]:
import os, sys, shutil, subprocess, time, json
CACHE = '/content/cache'
DRIVE_ROOT = '/content/drive/MyDrive/constrained-llm-js-codegen'
MS_DRIVE = f'{DRIVE_ROOT}/mask_store_zst'
os.makedirs(CACHE, exist_ok=True)
smi = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True)
GPU_NAME = smi.stdout.strip().splitlines()[0] if smi.returncode == 0 and smi.stdout.strip() else ''
print('GPU', GPU_NAME or 'none')
if not GPU_NAME or 'T4' in GPU_NAME:
    raise RuntimeError(f'This runtime has {GPU_NAME or "no GPU"}; pick G4, or L4 if G4 is not available (Runtime > Change runtime type).')
GPU = next((short for key, short in [('RTX PRO 6000', 'G4'), ('A100', 'A100'), ('H100', 'H100'), ('L4', 'L4')]
            if key in GPU_NAME), GPU_NAME.replace(' ', '_'))
from google.colab import drive
drive.mount('/content/drive')
REPO = '/content/code'
if os.path.isdir(REPO):
    shutil.rmtree(REPO)
for cmd in (['git', 'init', '-q', REPO], ['git', '-C', REPO, 'remote', 'add', 'origin', REPO_URL],
            ['git', '-C', REPO, 'fetch', '-q', '--depth', '1', 'origin', COMMIT],
            ['git', '-C', REPO, 'checkout', '-q', 'FETCH_HEAD']):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(' '.join(cmd) + ' failed:\n' + r.stderr)
def pip(*a):
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *a], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('pip install failed: ' + ' '.join(a) + '\n' + r.stderr[-2000:])
from importlib.metadata import version
if version('transformers') != '5.20.1':
    was = version('transformers')
    pip('transformers==5.20.1')
    if 'transformers' in sys.modules:
        raise RuntimeError(f'transformers was {was}; installed 5.20.1. Runtime > Restart session, then Run all.')
pip('lark', 'regex', 'networkx', 'fire', 'interegular', 'tqdm', 'datasets', 'jsonschema', 'psutil')
if subprocess.run([sys.executable, '-c', 'from syncode.parsers.itergen_parser import IGParser'],
                  cwd=REPO, capture_output=True).returncode != 0:
    pip('--no-deps', '--ignore-requires-python', '--force-reinstall',
        'git+https://github.com/structuredllm/syncode.git@4d6c110a028d3fb9cbb5f1a6ac6446b696f3eac4')
if not shutil.which('zstd'):
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'zstd'], capture_output=True)
ENV = dict(os.environ, SYNCODE_CACHE=CACHE + '/syncode/', HF_HOME=CACHE + '/hf_home', PYTHONPATH=REPO,
           PYTHONUNBUFFERED='1', TOKENIZERS_PARALLELISM='false', HF_HUB_DISABLE_PROGRESS_BARS='1')
print('ready: code', COMMIT[:7], '| GPU', GPU, '| node', shutil.which('node') is not None)

In [ ]:
def restore_mask_store(model):
    grammar = f'{REPO}/generators/grammars/javascript.lark'
    probe = subprocess.run([sys.executable, '-c',
        'import syncode.common as c; from syncode.parsers.grammars import Grammar\n'
        f'tok = c.load_tokenizer({model!r}); g = Grammar({grammar!r})\n'
        'print("MS_PATH=" + c.SYNCODE_CACHE + "mask_stores/" + type(tok).__name__ + '
        '"/grammar_strict_" + g.hash() + "_" + str(tok.vocab_size) + ".pkl")'],
        env=ENV, cwd=REPO, capture_output=True, text=True)
    lines = [l for l in probe.stdout.splitlines() if l.startswith('MS_PATH=')]
    if probe.returncode != 0 or not lines:
        raise RuntimeError('could not resolve the mask store path:\n' + probe.stderr[-2000:])
    path = lines[-1][len('MS_PATH='):]
    zst = f'{MS_DRIVE}/{os.path.basename(path)}.zst'
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if not (os.path.exists(path) and os.path.getsize(path) > 100 * 2**20):
        if not os.path.exists(zst):
            raise RuntimeError(f'mask store not on Drive: {zst}')
        r = subprocess.run(['zstd', '-q', '-d', '-f', '-T0', zst, '-o', path + '.part'], capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError('the copy on Drive is unreadable: ' + r.stderr[-300:])
        os.replace(path + '.part', path)
    print('mask store ready:', os.path.basename(path))

def status_of(run_dir):
    p = f'{run_dir}/eval.jsonl'
    return {json.loads(l)['name']: json.loads(l)['status'] for l in open(p)} if os.path.exists(p) else {}

quiet = ('Warning', 'warn(', 'Loading weights', 'torch_dtype', 'httpx')

def run_mode(model, mode, tasks):
    slug = model.split('/')[-1]
    os.makedirs('/content/work', exist_ok=True)
    tasks_file = f'/content/work/{slug}_{mode}_tasks.txt'
    open(tasks_file, 'w').write('\n'.join(tasks) + '\n')
    name = f'{mode}_{GPU}_{COMMIT[:7]}'
    out = f'/content/work/{slug}/{name}'
    r = subprocess.run([sys.executable, 'run.py', '--model', model, '--mode', mode, '--out', out,
                        '--tasks_file', tasks_file], cwd=REPO, env=ENV,
                       text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print('\n'.join(l for l in r.stdout.splitlines() if not any(s in l for s in quiet))[-1500:])
    if r.returncode != 0:
        raise RuntimeError(f'{slug} {mode}: run.py failed')
    subprocess.run([sys.executable, 'evaluate.py', out], cwd=REPO, capture_output=True, text=True)
    shutil.copytree(out, f'{DRIVE_ROOT}/final/{slug}/{name}', dirs_exist_ok=True)
    return out

PART1 = GPU == 'G4'
print('GPU', GPU, '->', 'Part 1 and Part 2' if PART1 else 'Part 2 only (Part 1 needs G4, the GPU of the report runs)')
try:
    for model in LOST:
        slug = model.split('/')[-1]
        print(f'\n######## {slug}')
        restore_mask_store(model)
        unc = status_of(f'{DRIVE_ROOT}/final/{slug}/unconstrained_{REPORT_TAG}')

        if PART1:
            out = run_mode(model, 'itergen_off', list(LOST[model]))
            now, rep = status_of(out), status_of(f'{DRIVE_ROOT}/final/{slug}/itergen_off_{REPORT_TAG}')
            print(f'\n== Part 1, {slug}: the lost tasks, checks off (v1 report vs v2 now, both G4)')
            print(f'{"task":32} {"class":6} {"unconstr.":15} {"v1 report":15} {"v2 now":15} masks fallback')
            for n, cls in LOST[model].items():
                s = json.load(open(f'{out}/{n}.json'))['stats']
                print(f'{n:32} {cls:6} {unc.get(n, "?"):15} {rep.get(n, "?"):15} {now.get(n, "?"):15} '
                      f'{s["mask_applied"] + s["mask_empty"]:<5} {s["fallback_used"]}')

        if HELPER.get(model):
            v2 = run_mode(model, 'itergen_on', HELPER[model])
            v3 = run_mode(model, 'itergen_on_repair', HELPER[model])
            s2, s3 = status_of(v2), status_of(v3)
            rep = status_of(f'{DRIVE_ROOT}/final/{slug}/itergen_on_{REPORT_TAG}')
            print(f'\n== Part 2, {slug}: checks on, v2 and v3 (helper repair) on {GPU}; unconstr. and v1 report from G4')
            print(f'{"task":34} {"unconstr.":15} {"v1 report":15} {"v2 now":15} {"v3 repair":15} helpers added / dropped')
            for n in HELPER[model]:
                st = json.load(open(f'{v3}/{n}.json'))['stats']
                print(f'{n:34} {unc.get(n, "?"):15} {rep.get(n, "?"):15} {s2.get(n, "?"):15} {s3.get(n, "?"):15} '
                      f'{st.get("helpers_added")} / {st.get("helpers_dropped")}')
            for n in HELPER[model]:
                d = json.load(open(f'{v3}/{n}.json'))
                print(f'\n--- v3 {n}  ({s3.get(n, "?")})')
                print(d['completion'].rstrip())
    print('\nALL DONE')
finally:
    if AUTO_DISCONNECT:
        from google.colab import drive, runtime
        drive.flush_and_unmount()
        time.sleep(30)
        runtime.unassign()
